# Exploratory notebook

Kept as run during the competition, with one change: the original absolute
paths on Stanford's Sherlock cluster have been replaced by `eegchallenge.config`
and the `EEGCHALLENGE_*` environment variables, as everywhere else in the
repository. `source config/paths.sh` before running.

The directories these cells originally read no longer exist (`$SCRATCH` was
purged), so the cells will not reproduce without regenerating the grid-search
output first. See the README section "Reproducibility notes".


In [ ]:
%load_ext memory_profiler
%config InlineBackend.figure_formats = {'svg',}


import joblib
from shutil import rmtree
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import signal
from matplotlib import pyplot as plt
import seaborn as sns
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from tqdm import tqdm

from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import FunctionTransformer,StandardScaler
from sklearn.model_selection import GridSearchCV,ParameterGrid
from sklearn.linear_model import SGDRegressor,RidgeCV,Ridge,LassoCV,ElasticNetCV

from eegchallenge import config

FROM = config.arrays_dir(challenge=1) / 'supervised_linear_grid_search'
TO = config.results_root() / 'supervised_linear'
FIGS_TO = config.results_root() / 'supervised_linear_results_figs'
TO.mkdir(parents=True, exist_ok=True)
FIGS_TO.mkdir(parents=True, exist_ok=True)
[i.unlink() for i in FIGS_TO.iterdir() if i.is_file()]
global FIG_I
FIG_I = 0


In [ ]:
def get_results_new(job_ids, remove_single_unique=False, save=True, test=False):
    df = []
    for job_id in job_ids:
        if not test:
            l = [i for i in (FROM/job_id).iterdir() if i.is_file()]
        else:
            l = [i for i in (FROM/job_id).iterdir() if i.is_file()][:1]
        for i in tqdm(l):
            try:
                result = joblib.load(i)
            except (ValueError,EOFError):
                continue
            df.append(pd.DataFrame(result.cv_results_))
    df = pd.concat(df)
    # Change object dtypes to string:
    for col in df:
        if df.loc[:,col].dtype==object:
            df.loc[:,col] = df.loc[:,col].astype('str')
    # Convert r2 to nrmse:
    df.loc[:,'nrmse_test'] = np.sqrt(1 - df.loc[:,'mean_test_score'])
    df.loc[:,'nrmse_train'] = np.sqrt(1 - df.loc[:,'mean_train_score'])
    # Shorten names:
    col_replace_replace_with = [
        ('param_frequency', 'FunctionTransformer(func=<function power_phase_circular at', 'power_phase_circular'),
        ('param_frequency', 'FunctionTransformer(func=<function power_phase_stft at', 'power_phase_stft'),
        ('param_feature_map', 'FunctionTransformer(func=<function polynomial at', 'polynomial'),
        ('param_frequency', 'FunctionTransformer(func=<function power_phase at', 'power_phase'),
        ('param_frequency', 'FunctionTransformer(func=<function bandpower_canonical at', 'bandpower_canonical'),
        ('param_frequency', 'FunctionTransformer(func=<function functional_connectivity at', 'functional_connectivity'), 
        ('param_frequency', 'FunctionTransformer(func=<function power_phase_temporal at', 'power_phase_temporal'),
        ('param_frequency', 'FunctionTransformer(func=<function enveloppe at', 'enveloppe'),
        ('param_frequency', 'FunctionTransformer(func=<function complex_cepstrum at', 'complex_cepstrum'),
        ('param_frequency', 'FunctionTransformer(func=<function real_cepstrum', 'real_cepstrum'),
        ('param_scale', 'FunctionTransformer(func=<function mad_scaling_clipping_channel_time at', 'mad_scaling_clipping_channel_time'),
        ('param_scale', 'FunctionTransformer(func=<function mad_scaling_clipping_channel at', 'mad_scaling_clipping_channel'),
        ('param_scale', 'FunctionTransformer(func=<function mad_scaling_clipping_channel_trial at', 'mad_scaling_clipping_channel_trial'),
        ('param_first_scale', 'FunctionTransformer(func=<function mad_scaling_clipping_trial_time at', 'mad_scaling_clipping_trial_time'),
        ('param_first_scale', 'FunctionTransformer(func=<function mad_scaling_clipping_channel_trial at', 'mad_scaling_clipping_channel_trial'),
        ('param_model', "Ridge(solver='lsqr')", 'Ridge'),
        ('param_model', 'HistGradientBoostingRegressor(early_stopping=True, max_iter=300,', 'HistGradientBoostingRegressor'),
        ('param_model', 'ExtraTreesRegressor(min_samples_leaf=20, n_estimators=400, n_jobs=1,', 'ExtraTreesRegressor'),
    ]
    for col, replace, replace_with in col_replace_replace_with:
        assert replace_with in replace
        if col in df.columns:
            df.loc[df.loc[:,col].str.contains(replace,regex=False), col] = replace_with
    assert df.loc[:,'nrmse_test'].notna().all()
    assert df.loc[:,'nrmse_train'].notna().all()
    if remove_single_unique:
        df = df.loc[:, df.nunique() > 1]
    if save:
        df.to_csv(TO/f"{'_'.join(job_ids)}.csv")
    return df


def save_fg(fg):
    global FIG_I
    fg.savefig(FIGS_TO/f'{FIG_I}.png',dpi=1e3)
    FIG_I += 1

# 11-02

## Multi-layer perceptron

In [ ]:
RE_COLLECT = True
if RE_COLLECT:
    df = get_results_new(['8924180'])

In [ ]:
fg = sns.catplot(df, y='nrmse_test')

## Get best model for each feature set

In [ ]:
df = []
for i in ['8783179','8783173','8783182','8791474']:
    df.append(pd.read_csv(TO/f'{i}.csv',index_col=0))
df = pd.concat(df, ignore_index=True)

In [ ]:
# Best power_phase_circular
df.loc[df.query('param_frequency=="power_phase_circular" & param_first_scale=="passthrough" & param_second_scale=="passthrough"').loc[:,'nrmse'].idxmin()]

In [ ]:
# Best enveloppe
df.loc[df.query('param_frequency=="enveloppe" & param_first_scale=="passthrough" & param_second_scale=="passthrough"').loc[:,'nrmse'].idxmin()]

## Temporal and STFT grid search results

In [ ]:
# JOB_IDS = ['8897851', '8898187', '8899909', '8904276', '8904275', '8904260'] TODO: Uncomment when 8904276 LJ STFT are finished
JOB_IDS = ['8897851', '8898187', '8899909', '8904275', '8904260']


In [ ]:
RE_COLLECT = True

if RE_COLLECT:
    for id in JOB_IDS:
        df = get_results_new([id])

In [ ]:
# Combine each job's results
df = []
for i in JOB_IDS:
    df.append(pd.read_csv(TO/f'{i}.csv',index_col=0))
df = pd.concat(df, ignore_index=True)

### Overall

In [ ]:
fg = sns.catplot(df, col='param_frequency', y='nrmse_test', col_order=['passthrough','power_phase_stft'])
save_fg(fg)

### Test versus train

In [ ]:
df_long = df.melt(id_vars=['param_frequency'],value_vars=['nrmse_test','nrmse_train'],var_name='set',value_name='nrmse')

In [ ]:
fg = sns.catplot(df_long, col='param_frequency', y='nrmse', hue='set', dodge=True, col_order=['passthrough','power_phase_stft'])
save_fg(fg)

In [ ]:
fg = sns.relplot(df, x='nrmse_train', y='nrmse_test', col='param_frequency', col_order=['passthrough','power_phase_stft'])
save_fg(fg)

### Overfitting by parameter value

In [ ]:
df.loc[:,'test-train'] = df.loc[:,'nrmse_test'] - df.loc[:,'nrmse_train']
hyper_params = ['model__learning_rate', 'model__l2_regularization', 'model__max_leaf_nodes', 'model__min_samples_leaf', 'model__max_features', 'model__max_bins', 'model__max_depth']

In [ ]:
for param in hyper_params:
    fg = sns.relplot(df, x=f'param_{param}', y='test-train', col='param_frequency', col_order=['passthrough','power_phase_stft'])
    save_fg(fg)

### Test NRMSE by parameter value

In [ ]:
for param in hyper_params:
    fg = sns.relplot(df, x=f'param_{param}', y='nrmse_test', col='param_frequency', col_order=['passthrough','power_phase_stft'])
    save_fg(fg)

In [ ]:
raise SystemExit

# 11-01

In [ ]:
JOB_IDS = ['8865913','8867010',]

In [ ]:
RE_COLLECT = False

if RE_COLLECT:
    for id,shape in [(JOB_IDS[0],108), (JOB_IDS[1],107),]:
        df = get_results_new([id])
        # assert df.shape[0]==(shape*2)

In [ ]:
df = []
for i in JOB_IDS:
    df.append(pd.read_csv(TO/f'{i}.csv',index_col=0))
df = pd.concat(df, ignore_index=True)

df.loc[:,'nrmse_test'] = np.sqrt(1 - df.loc[:,'mean_test_score'])
df.loc[:,'nrmse_train'] = np.sqrt(1 - df.loc[:,'mean_train_score'])
df.loc[:,'test-train'] = df.loc[:,'nrmse_test'] - df.loc[:,'nrmse_train']
df_long = df.melt(id_vars=['param_frequency'],value_vars=['nrmse_test','nrmse_train'],var_name='set',value_name='nrmse')

In [ ]:
n_passthrough = df.query('param_frequency=="passthrough"').shape[0]
n_stft = df.query('param_frequency=="power_phase_stft"').shape[0]
assert (n_passthrough+n_stft)==df.shape[0]

print(f'Passthrough:',f'{100*n_passthrough/1000}% finished')
print(f'STFT:',f'{100*n_stft/1000}% finished')

## Overall

In [ ]:
fg = sns.catplot(df, col='param_frequency', y='nrmse_test', dodge=True, col_order=['passthrough','power_phase_stft'])
save_fg(fg)

## Test versus train

In [ ]:
fg = sns.catplot(df_long, col='param_frequency', y='nrmse', hue='set', dodge=True, col_order=['passthrough','power_phase_stft'])
save_fg(fg)

In [ ]:
fg = sns.relplot(df, x='nrmse_train', y='nrmse_test', col='param_frequency', col_order=['passthrough','power_phase_stft'])
save_fg(fg)

## Overfitting by parameter value

In [ ]:
for param in ['model__learning_rate', 'model__l2_regularization', 'model__max_leaf_nodes', 'model__min_samples_leaf', 'model__max_features', 'model__max_bins', 'model__max_depth']:
    fg = sns.relplot(df, x=f'param_{param}', y='test-train', col='param_frequency', col_order=['passthrough','power_phase_stft'])
    save_fg(fg)

## Test NRMSE by parameter value

In [ ]:
for param in ['model__learning_rate', 'model__l2_regularization', 'model__max_leaf_nodes', 'model__min_samples_leaf', 'model__max_features', 'model__max_bins', 'model__max_depth']:
    fg = sns.relplot(df, x=f'param_{param}', y='nrmse_test', col='param_frequency', col_order=['passthrough','power_phase_stft'])
    save_fg(fg)

In [ ]:
raise SystemExit

# 10-31

In [ ]:
# TODO: Maybe figure out why '8783173' does not have 108 .pkls
# TODO: Maybe figure out why '8783182' does not have 72 .pkls

# for id,shape in [('8783179',108), ('8783173',107), ('8783182',71)]:
#     df = get_results_new([id])
#     assert df.shape[0]==(shape*2)

In [ ]:
def get_combined_df():

    df = []
    for i in ['8783179','8783173','8783182']:
        df.append(pd.read_csv(TO/f'{i}.csv',index_col=0))
    df = pd.concat(df)

    mask1 = df.loc[:,'param_frequency'].isin(['power_phase_circular','power_phase_stft'])
    mask2 = df.loc[:,'param_feature_map'].isin(['polynomial'])
    assert (mask1 ^ mask2).all()
    df.loc[mask1,'features'] = df.loc[mask1,'param_frequency']
    df.loc[mask2,'features'] = df.loc[mask2,'param_feature_map']

    return df


df = get_combined_df()

## Effect of `power_phase_circular`, `power_phase_stft`, `polynomial`

In [ ]:
fg = sns.catplot(df.query('nrmse<1'), x='features', y='nrmse', col='param_model', hue='param_first_scale', col_order=['HistGradientBoostingRegressor','Ridge'], dodge=True)
fg = sns.catplot(df.query('nrmse<1'), x='features', y='nrmse', col='param_model', hue='param_second_scale', col_order=['HistGradientBoostingRegressor','Ridge'], dodge=True)
fg.tick_params(rotation=90, axis='x',)

## Effect of `contrast`

In [ ]:
l = []
for i in [i for i in (FROM/'8783187').iterdir() if i.is_file()]:
    try:
        joblib.load(i)
        l.append('success')
    except AttributeError:
        l.append('failure')
assert (np.array(l)=='failure').all()

In [ ]:
import os
from pathlib import Path
import re
from typing import List

def collect_contrast_scores(
    dir_path: str = None
) -> List[float]:
    """
    Iterate over each file in `dir_path`. For files whose names contain 'contrast',
    read lines and extract numbers following 'score=' (e.g., 'score=-14.588').
    Returns a list of all parsed scores as floats.

    Defaults to $EEGCHALLENGE_JOBS, the Slurm log directory set in
    config/paths.sh.
    """
    if dir_path is None:
        dir_path = os.environ['EEGCHALLENGE_JOBS']
    p = Path(dir_path)
    if not p.is_dir():
        raise NotADirectoryError(f"Not a directory: {dir_path}")

    # Matches: score= -12, score=+3.14, score=2.5e-3, with optional spaces
    score_re = re.compile(r"score\s*=\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)")

    scores: List[float] = []
    for entry in p.iterdir():
        if not entry.is_file():
            continue
        if "contrast" not in entry.name.lower():
            continue

        # Read text; ignore decoding errors so we can scan any odd files.
        try:
            with entry.open("r", encoding="utf-8", errors="ignore") as fh:
                for line in fh:
                    for m in score_re.finditer(line):
                        try:
                            scores.append(float(m.group(1)))
                        except ValueError:
                            # In case of an odd parse error, skip that match.
                            continue
        except (OSError, IOError):
            # Skip unreadable files
            continue

    return scores


scores = collect_contrast_scores()
new_rows = {"features": len(scores)*['contrast'], "nrmse": np.sqrt(1 - np.array(scores))}
df = pd.concat([df, pd.DataFrame(new_rows)])

In [ ]:
fg = sns.catplot(df.query('nrmse<1'), x='features', y='nrmse', dodge=True)
fg.tick_params(rotation=90, axis='x',)

In [ ]:
raise SystemExit

## Fit times

In [ ]:
# df = get_results_new(['8791474'])
df = pd.read_csv(TO/'8791474.csv',index_col=0)

In [ ]:
sns.catplot(df, x='param_model', y='mean_fit_time')
plt.tick_params(axis='x',rotation=90)

## Overall performance by model

In [ ]:
plt.subplot(1,2,1)
sns.stripplot(df, x='param_model', y='nrmse')
plt.tick_params(axis='x',rotation=90)

plt.subplot(1,2,2)
sns.stripplot(df.query('nrmse<1'), x='param_model', y='nrmse')
plt.tick_params(axis='x',rotation=90)

## Effect of frequency transform, first scaling, second scaling (grid_search)
- Frequency transform
  - Passthrough is still best, with envelope about the same
- First scaling
  - Passthrough is still best overall
  - Scaling over channels is best for: `power_phase`
- Second scaling
  - This does not make a difference
  - I'm surprised it's apparently worsening Ridge performance

In [ ]:
fg = sns.catplot(df.query('nrmse<1'), x='param_frequency', y='nrmse', col='param_model', hue='param_first_scale', dodge=True, order=['passthrough','enveloppe','power_phase','complex_cepstrum','real_cepstrum'], hue_order=['passthrough','mad_scaling_clipping_trial_time','mad_scaling_clipping_channel_trial'])
fg.tick_params(rotation=90, axis='x')

fg = sns.catplot(df.query('nrmse<1'), x='param_frequency', y='nrmse', col='param_model', hue='param_second_scale', dodge=True, order=['passthrough','enveloppe','power_phase','complex_cepstrum','real_cepstrum'], hue_order=['passthrough','RobustScaler(copy=False)'])
fg.tick_params(rotation=90, axis='x')

In [ ]:
raise SystemExit

# 10-30

In [ ]:
df = get_results_new(['8681230',])

## Fit times

In [ ]:
sns.catplot(df, x='param_model', y='mean_fit_time')
plt.tick_params(axis='x',rotation=90)

## Overall performance by model

In [ ]:
plt.subplot(1,2,1)
sns.stripplot(df, x='param_model', y='nrmse')
plt.tick_params(axis='x',rotation=90)

plt.subplot(1,2,2)
sns.stripplot(df.query('nrmse<1'), x='param_model', y='nrmse')
plt.tick_params(axis='x',rotation=90)

## Effect of frequency transform, first scaling, second scaling (grid_search)
- Frequency transform
  - Passthrough is still best, with envelope about the same
- First scaling
  - Passthrough is still best, with scaling over channels or over time similar for some frequency transforms
- Second scaling
  - This does not make a difference
  - I'm surprised it's apparently worsening Ridge performance

In [ ]:
fg = sns.catplot(df.query('nrmse<1'), x='param_frequency', y='nrmse', col='param_model', hue='param_first_scale', dodge=True, order=['passthrough','enveloppe','power_phase','complex_cepstrum','real_cepstrum'], hue_order=['passthrough','mad_scaling_clipping_trial_time','mad_scaling_clipping_channel_trial'])
fg.tick_params(rotation=90, axis='x')

fg = sns.catplot(df.query('nrmse<1'), x='param_frequency', y='nrmse', col='param_model', hue='param_second_scale', dodge=True, order=['passthrough','enveloppe','power_phase','complex_cepstrum','real_cepstrum'])
fg.tick_params(rotation=90, axis='x')

In [ ]:
raise SystemExit

# 10-29

In [ ]:
df = []
for job_id in ['8640608','8580353','8581174','8582532','8581062','8660282']:
    for file in [i for i in (FROM / job_id).iterdir() if i.is_file()]:
        df.append(pd.DataFrame(joblib.load(file).cv_results_))
df = pd.concat(df)
df.loc[:,'param_model'] = df.loc[:,'param_model'].astype('str')

sns.catplot(df, x='param_model', y='mean_fit_time')
plt.tick_params(axis='x',rotation=90)

# 10-28

## Functional connectivity is not good

- Functional connectivity is never better than chance (and worse than passthrough and power + phase, especially clearly for ensembling models)

In [ ]:
df = get_results('8520196')

In [ ]:
fg = sns.catplot(df.query('nrmse < 1.5'), col='param_model', hue='param_first_scale', x='param_frequency', y='nrmse', kind='swarm', size=2.5, warn_thresh=0)
for ax in fg.axes.flatten():
    ax.axhline(1,ls='--',c='grey')
fg = sns.catplot(df.query('nrmse < 1.5'), col='param_model', x='param_first_scale', hue='param_frequency', y='nrmse', kind='swarm', size=2.5, warn_thresh=0)
for ax in fg.axes.flatten():
    ax.axhline(1,ls='--',c='grey')

In [ ]:
plt.figure(figsize=(10,14))
sns.swarmplot(df.query('nrmse<1'), y='param_second_scale', x='nrmse', size=3, warn_thresh=0)
sns.barplot(df.query('nrmse<1'), y='param_second_scale', x='nrmse', )
sns.barplot(df.query('nrmse<1'), y='param_second_scale', x='nrmse', estimator=np.min)
plt.xlim(left=0.9)

## Combining power_phase and temporal is no better than either alone

- Also notable: Results suggest that adding irrelevant features not hurting performance (since power_phase_temporal the same as temporal)

In [ ]:
df2 = get_results('8520950', remove_single_unique=False)
df = pd.concat((df,df2),axis=0)

In [ ]:
fg = sns.catplot(df.query('nrmse < 1.5'), col='param_model', hue='param_first_scale', x='param_frequency', y='nrmse', kind='swarm', size=1.75, warn_thresh=0, order=['passthrough','power_phase_temporal','power_phase','functional_connectivity'])
for ax in fg.axes.flatten():
    ax.axhline(1,ls='--',c='grey')
    ax.tick_params(rotation=45,axis='x')
fg = sns.catplot(df.query('nrmse < 1.5'), col='param_model', x='param_first_scale', hue='param_frequency', y='nrmse', kind='swarm', size=1.75, warn_thresh=0, hue_order=['passthrough','power_phase_temporal','power_phase','functional_connectivity'])
for ax in fg.axes.flatten():
    ax.axhline(1,ls='--',c='grey')

## Ridge performance improves with regularization strength (alpha size)

In [ ]:
plt.figure(figsize=(10,10))
ax = sns.scatterplot(df.query('param_model=="Ridge"'), x='param_model__alpha', y='nrmse')
ax.set(xscale="log")
ax.axhline(1,ls='--',c='grey')

# 10-27

In [ ]:
df = get_results('8441740')

## Frequency = passthrough gives best and worst results
- `power_phase`gives decent results
- `bandpower_canonical` is not working -- not better than chance 

In [ ]:
plt.figure(figsize=(22,10))

plt.subplot(1,2,1)
sns.swarmplot(df.query('nrmse<1.5'),y='nrmse',hue='param_frequency',x='param_scale',warn_thresh=0,size=4)
plt.ylim(bottom=0.9)
plt.tick_params(axis='x',labelrotation=90)
plt.axhline(1,ls='--',c='grey')

plt.subplot(1,2,2)
sns.swarmplot(df.query('nrmse<1.5'),y='nrmse',x='param_frequency',hue='param_scale',warn_thresh=0,size=4)
plt.ylim(bottom=0.9)
plt.tick_params(axis='x',labelrotation=90)
plt.axhline(1,ls='--',c='grey')

## How do model, scaling, and frequency engineering affect performance?

Model
- Ensembling models are best
- HistGradientBoostingRegressor > ExtraTreesRegressor ≈ PLSRegression = Ridge

Scaling
- For ensembling models: Scaling does not matter
- For linear models: scaling per channel and trial is best

In [ ]:
g = df.groupby(['param_model','param_scale','param_frequency'])['nrmse'].min()
vmin, vmax = float(g.min()), float(g.max())

for model, d in df.groupby("param_model"):
    mat = d.pivot_table(index="param_scale",
                        columns="param_frequency",
                        values="nrmse",
                        aggfunc="min")
    plt.figure(figsize=(6,4))
    sns.heatmap(mat, annot=True, fmt=".3f",
                vmin=vmin, vmax=vmax, cbar_kws={"label":"Best NRMSE ↓"})
    plt.title(f"{model}: best NRMSE per (param_scale × param_frequency)")
    plt.xlabel("param_frequency"); plt.ylabel("param_scale")


In [ ]:
plt.figure(figsize=(24,10))

plt.subplot(1,2,1)
sns.swarmplot(df.query('nrmse<1.5'),y='nrmse',x='param_model',hue='param_frequency',warn_thresh=0,size=3)
plt.ylim(bottom=0.9)
plt.tick_params(axis='x',labelrotation=90)
plt.axhline(1,ls='--',c='grey')

plt.subplot(1,2,2)
sns.swarmplot(df.query('nrmse<1.5'),y='nrmse',x='param_model',hue='param_scale',warn_thresh=0,size=3)
plt.ylim(bottom=0.9)
plt.tick_params(axis='x',labelrotation=90)
plt.axhline(1,ls='--',c='grey')